In [ ]:
!git clone https://github.com/ChrisSkjer/energy-stable-pinns-ch.git
%cd energy-stable-pinns-ch

In [ ]:
!pip install -r requirements-colab.txt

In [ ]:
import torch
print(torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Flags for `python -m src.pinn.train` (defaults shown; patience 0 = off):
```
model    --hidden-layers 5  --hidden-width 100
adam     --epochs 100  --lr 1e-3  --adam-tol 1e-4  --adam-patience 0
plateau  --plateau-patience 0  --plateau-factor 0.5  --min-lr 1e-6
lbfgs    --lbfgs-steps 70  --lbfgs-tol 1e-6  --lbfgs-patience 3
weights  --pde-u-weight 1  --pde-mu-weight 100  --ic-weight 100  --bc-weight 10
energy   --energy-penalty (switch)  --energy-weight 1
problem  --epsilon 0.05  --x-max 1  --y-max 1  --t-max 5e-3
points   --n-collocation 10000  --n-ic 100  --n-bc 100  --pde-at-t0 2000
run      --device cpu  --run-name <timestamp>  --checkpoint-every 500  --log-every 100  --smoke-test
```

In [ ]:
# One run per initial condition, back to back. Each run gets its own run name
# (base + _ic-<name>) so it lands in its own folder under results/pinn_models/.
# --smoke-test runs a handful of iterations on a tiny point set to check the
# pipeline end to end; drop it and set your own flags for a real run.
# Tip: copy this notebook to notebooks/train_pinn_local.ipynb (gitignored) for
# your own runs so outputs and flag tweaks stay out of the repo.
base_name = "smoke"
ics = ["cross", "two_circles", "noise"]
run_names = {ic: f"{base_name}_ic-{ic}" for ic in ics}

for ic, run_name in run_names.items():
    print(f" ########## training ic={ic} -> {run_name} ##########")
    !python -m src.pinn.train --run-name {run_name} --ic {ic} --device cuda --smoke-test

In [ ]:
# NOTE: google.colab.files.download() needs the actual colab.research.google.com
# frontend to receive the browser download -- it silently does nothing when this
# notebook is driven from VS Code's Jupyter extension against a Colab kernel.
# Instead, embed each model as base64 text in the cell output, which VS Code
# stores locally in this .ipynb file once you save it (Ctrl+S). Then run
# `python scripts/pull_colab_model.py` locally to decode them -- every run
# goes to results/pinn_models/<run_name>/final.pt.
import base64
import pathlib

for run_name in run_names.values():
    model_path = pathlib.Path(f"results/pinn_models/{run_name}/final.pt")
    if not model_path.exists():
        print(f"skipping {run_name}: no final.pt (run failed?)")
        continue
    b64 = base64.b64encode(model_path.read_bytes()).decode()
    print(f"===BEGIN_MODEL_B64:{run_name}/final.pt===")
    print(b64)
    print("===END_MODEL_B64===")